# Receipt Field Extraction — LoRA Fine-tuning on Colab T4

Fine-tunes **Qwen2.5-3B-Instruct** with QLoRA (4-bit + rank-8 LoRA) using HuggingFace PEFT + TRL.

**Before running:** set Runtime → Change runtime type → T4 GPU.

In [ ]:
# Verify T4 GPU is attached
!nvidia-smi

## 1. Install dependencies

In [ ]:
!pip install -q \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    trl>=0.8.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.28.0 \
    datasets>=2.19.0 \
    huggingface_hub>=0.23.0 \
    rapidfuzz \
    tqdm

## 2. HuggingFace login

Needed to download Qwen2.5-3B-Instruct and push adapters back to Hub.
Get your token from https://huggingface.co/settings/tokens (write access).

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Reads HF_TOKEN from Colab Secrets (Secrets tab in left sidebar → add HF_TOKEN)
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

## 3. Get the repo and data

**Option A — clone from GitHub** (recommended if repo is pushed):
```
!git clone https://github.com/<your-username>/<repo-name>.git receipt-extraction
%cd receipt-extraction
```

**Option B — mount Google Drive** (if data files are there):
```python
from google.colab import drive
drive.mount('/content/drive')
# Then copy: !cp -r /content/drive/MyDrive/receipt-extraction /content/
```

Run whichever block fits your setup:

In [ ]:
# Option A: clone from GitHub
GITHUB_REPO = "https://github.com/<your-username>/<repo-name>.git"  # ← fill in

!git clone {GITHUB_REPO} receipt-extraction
%cd receipt-extraction

In [ ]:
# Option B: mount Google Drive instead
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/receipt-extraction /content/
# %cd /content/receipt-extraction

## 4. Prepare data

If `data/final/mlx_format/train.jsonl` already exists (copied from Drive), skip to step 5.

Otherwise run the data pipeline to regenerate from scratch:

In [ ]:
import os

TRAIN_PATH = "data/final/mlx_format/train.jsonl"

if os.path.exists(TRAIN_PATH):
    print(f"Data already present: {TRAIN_PATH} — skipping regeneration")
else:
    print("Regenerating data from scratch...")

    # Download and process SROIE (no API key needed)
    !python prepare_sroie.py

    # Generate synthetic Indian receipts (requires OPENAI_API_KEY)
    # Uncomment if you have an OpenAI key in Colab Secrets:
    # import os
    # os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    # !python generate_synthetic.py

    # Merge datasets + stratified split
    !python merge_datasets.py

print("Data ready.")

In [ ]:
# Confirm split sizes
import json

for split in ["train", "valid", "test"]:
    path = f"data/final/mlx_format/{split}.jsonl"
    with open(path) as f:
        n = sum(1 for _ in f)
    print(f"{split:6s}: {n} examples")

## 5. Train

Runs QLoRA fine-tuning: rank=8, alpha=16, last 16 of 36 Qwen layers, 10 epochs, effective batch=8.
Expected runtime on T4: **~45–60 minutes**.

In [ ]:
!python train_peft.py \
    --epochs 10 \
    --batch-size 4 \
    --grad-accum 2 \
    --output-dir ./adapters

## 6. Evaluate

In [ ]:
# Baseline (no adapters)
!python baseline_eval.py --output ./results/baseline_results.json

In [ ]:
# Fine-tuned
!python baseline_eval.py \
    --adapter-path ./adapters \
    --output ./results/finetuned_results.json

In [ ]:
!python compare_results.py

## 7. Push adapters to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi

REPO_ID = "largetrader/qwen2.5-3b-receipt-extraction-lora"  # ← your HF repo

api = HfApi()
api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path="./adapters",
    repo_id=REPO_ID,
    repo_type="model",
)
print(f"Adapters pushed → https://huggingface.co/{REPO_ID}")

## 8. (Optional) Save adapters to Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r ./adapters /content/drive/MyDrive/receipt-extraction-adapters
# print("Adapters saved to Google Drive.")